In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/task1_model_table.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(2593, 10)
['zip', 'total_fire_count', 'total_acres', 'fire_year_count', 'avg_tmax_c', 'avg_tmin_c', 'avg_prcp_mm', 'target_2023', 'fire_2022', 'rolling_3yr_fire_count']


,zip,total_fire_count,total_acres,fire_year_count,avg_tmax_c,avg_tmin_c,avg_prcp_mm,target_2023,fire_2022,rolling_3yr_fire_count
0,90001,0.0,0.0,0,22.198906,13.309137,289.50,0,0,0.0
1,90002,0.0,0.0,0,22.198906,13.309137,289.50,0,0,0.0
2,90003,0.0,0.0,0,22.198906,13.309137,289.50,0,0,0.0
3,90004,0.0,0.0,0,24.746478,12.360862,313.98,0,0,0.0
4,90005,0.0,0.0,0,22.198906,13.309137,289.50,0,0,0.0


In [2]:
X = df.drop(columns=['zip', 'target_2023'])
y = df['target_2023']

print(X.shape)
print(y.value_counts())

(2593, 8)
target_2023
0    2477
1     116
Name: count, dtype: int64


In [3]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, y_train.value_counts().to_dict())
print("Test:", X_test.shape, y_test.value_counts().to_dict())

Train: (2074, 8) {0: 1981, 1: 93}
Test: (519, 8) {0: 496, 1: 23}


In [5]:
from sklearn.preprocessing import StandardScaler

# log1p convert?
X_train['total_acres_log'] = np.log1p(X_train['total_acres'])
X_test['total_acres_log'] = np.log1p(X_test['total_acres'])
X_train = X_train.drop(columns=['total_acres'])
X_test = X_test.drop(columns=['total_acres'])

# StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Done! Shape:", X_train_scaled.shape)

Done! Shape: (2074, 8)


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)

pred = model.predict(X_test_scaled)
prob = model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, pred))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 3))

              precision    recall  f1-score   support

           0       0.99      0.86      0.92       496
           1       0.20      0.74      0.31        23

    accuracy                           0.86       519
   macro avg       0.59      0.80      0.62       519
weighted avg       0.95      0.86      0.89       519

ROC-AUC: 0.906


In [7]:
pip install matplotlib


Note: you may need to restart the kernel to use updated packages.


In [8]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, pred)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[428  68]
 [  6  17]]
